In [1]:
!pip install sentencepiece datasets -q

In [2]:
import os
from datasets import load_dataset
import sentencepiece as spm

print('SentencePiece version:', spm.__version__)

## Prepare Training Text

In [3]:
dataset = load_dataset('wikitext', 'wikitext-2-raw-v1', split='train[:5%]')
print('Samples:', len(dataset))

train_file = 'spm_train.txt'
with open(train_file, 'w', encoding='utf-8') as f:
    for row in dataset['text']:
        if row.strip():
            f.write(row.strip() + '\n')
print('Wrote training text to', train_file)

## Train SentencePiece (BPE)

In [4]:
model_prefix = 'toy_bpe'
vocab_size = 800

spm.SentencePieceTrainer.Train(
    input=train_file,
    model_prefix=model_prefix,
    vocab_size=vocab_size,
    model_type='bpe',
    character_coverage=1.0,
    pad_id=0, unk_id=1, bos_id=2, eos_id=3
)

print('Trained files:', model_prefix + '.model', model_prefix + '.vocab')
print('Vocab size (requested):', vocab_size)

## Encode / Decode Examples

In [5]:
sp = spm.SentencePieceProcessor(model_file=model_prefix + '.model')

samples = [
    'Transformers changed NLP.',
    'SentencePiece handles languages without whitespace.',
    'Tokenizers control vocabulary size and OOV handling.'
]

for text in samples:
    ids = sp.encode(text, out_type=int)
    pieces = sp.encode(text, out_type=str)
    decoded = sp.decode(ids)
    print('\nOriginal:', text)
    print('Pieces:', pieces)
    print('IDs   :', ids)
    print('Decoded:', decoded)

## Save + Reload Check

In [6]:
# Re-load processor and ensure deterministic behavior
sp2 = spm.SentencePieceProcessor(model_file=model_prefix + '.model')
ids = sp2.encode(samples[0], out_type=int)
print('Reloaded encoding matches original?', ids == sp.encode(samples[0], out_type=int))